# Vehicle specification extraction - step-by-step walkthrough

Pipeline: **PDF -> layout-aware parsing -> structure-aware chunks -> embeddings (FAISS) + BM25 -> retrieve -> LLM / rule extraction -> JSON/CSV**

Each section below runs one stage of the pipeline so you can inspect its output. Set the three variables in the first cell
(PDF path, embedding backend, LLM provider) and run all cells.

In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath(".."))          # run from notebooks/ without installing

PDF   = os.getenv("SPECX_PDF", "../data/service-manual.pdf")
EMBED = os.getenv("SPECX_EMBED_BACKEND", "sentence-transformers")   # or "tfidf" (offline, no model download)
LLM   = os.getenv("SPECX_LLM_PROVIDER", "auto")                     # auto | anthropic | openai | none
CACHE = os.getenv("SPECX_CACHE", "../.specx_cache")

from specx.config import Settings
s = Settings(pdf_path=PDF, embed_backend=EMBED, llm_provider=LLM, cache_dir=CACHE)
print("PDF:", s.pdf_path, "| embeddings:", s.embed_backend, "| LLM:", s.resolved_llm_provider())

PDF: ../data/service-manual.pdf | embeddings: tfidf | LLM: none


## 1. PDF text extraction
Plain `page.get_text()` is not enough for this manual: the stream is out of reading order, tables come out one cell per line,
and every page carries print noise. `inspect` shows what the layout-aware loader produces for one page (page 636 = *Front Disc Brake - Specifications*).

In [2]:
from specx.pdf_loader import load_manual
page = load_manual(PDF, s, pages=range(635, 637), progress=False)
for d in page.docs:
    print(f"# section={d.section_label!r} type={d.doc_type!r} file={d.file_id} pages={d.page_start}-{d.page_end}")
    for e in d.elements:
        tag = f"table:{e.table_kind}" if e.kind == "table" else e.kind
        print(f"--- [{tag}] p{e.page}"); print(e.text)

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


# section='206-03 Front Disc Brake' type='SPECIFICATIONS' file=SE263001 pages=636-637
--- [heading] p636
Material
--- [table:material] p636
| Item | Specification | Fill Capacity |
|---|---|---|
| High Performance DOT 3 Motor Vehicle Brake Fluid PM-1-C (US); CPM-1-C (Canada) | WSS-M6C62-A or WSS-M6C65-A1 | 1,420 ml (3 pt) |
| Metal Brake Parts Cleaner PM-4-A or PM-4-B (US); CPM-4 (Canada) | — | — |
--- [heading] p636
General Specifications
--- [table:general] p636
| Item | Specification |
|---|---|
| Brake Disc |  |
| Brake disc minimum thickness | 32 mm (1.259 in) |
| Brake Pads |  |
| Brake pad maximum taper wear (in any direction) | 3.0 mm (0.118 in) |
| Brake pad minimum thickness | 3.0 mm (0.118 in) |
--- [heading] p636
Torque Specifications
--- [table:torque] p636
| Description | Nm | lb-ft | lb-in |
|---|---|---|---|
| Brake caliper anchor plate bolts | 250 | 184 | — |
| Brake caliper flow bolt | 35 | 26 | — |
| Brake caliper guide pin bolts | 37 | 27 | — |
| Brake disc shield b

## 2. Chunking
Spec tables become one chunk each; procedures are split at component / phase headings. Every chunk starts with a *breadcrumb*
(`[section | document type | heading path | pages]`) that is embedded with it.

In [3]:
from specx.chunker import build_chunks
chunks = build_chunks(page, s)
for c in chunks:
    print(c.id, c.chunk_type, "pages", c.page_start, "\n" + c.text[:420], "\n")

d000-00 material_table pages 636 
[206-03 Front Disc Brake | Specifications | Material: fluids, lubricants, sealants - part numbers, specification codes, fill capacity | p.636]
| Item | Specification | Fill Capacity |
|---|---|---|
| High Performance DOT 3 Motor Vehicle Brake Fluid PM-1-C (US); CPM-1-C (Canada) | WSS-M6C62-A or WSS-M6C65-A1 | 1,420 ml (3 pt) |
| Metal Brake Parts Cleaner PM-4-A or PM-4-B (US); CPM-4 (Canada) | — | — | 

d000-01 general_table pages 636 
[206-03 Front Disc Brake | Specifications | General Specifications | p.636]
| Item | Specification |
|---|---|
| Brake Disc |  |
| Brake disc minimum thickness | 32 mm (1.259 in) |
| Brake Pads |  |
| Brake pad maximum taper wear (in any direction) | 3.0 mm (0.118 in) |
| Brake pad minimum thickness | 3.0 mm (0.118 in) | 

d000-02 torque_table pages 636 
[206-03 Front Disc Brake | Specifications | Torque Specifications (Nm, lb-ft, lb-in) | p.636]
| Description | Nm | lb-ft | lb-in |
|---|---|---|---|
| Brake caliper anch

## 3. Index the whole manual (parse -> chunk -> embed -> FAISS + BM25)
The first run parses all pages (~45 s) and caches everything under `CACHE`.

In [4]:
import collections
from specx.pipeline import Pipeline
pipe = Pipeline(s)
pipe.ensure_ready()
print(len(pipe.manual.docs) if pipe.manual else "loaded from cache", "documents;", len(pipe.chunks), "chunks")
print(collections.Counter(c.chunk_type for c in pipe.chunks))

loaded from cache documents; 836 chunks
Counter({'text': 686, 'parts_table': 67, 'material_table': 54, 'torque_table': 19, 'general_table': 10})


## 4. Retrieval-augmented query
Hybrid retrieval (dense + BM25, fused with RRF, intent-aware boost):

In [5]:
QUERY = "Torque for brake caliper bolts"
from specx.query import analyze_query
info = analyze_query(QUERY)
print("intent:", info.intent, "| content terms:", info.terms)
for h in pipe.retriever.search(QUERY, k=6):
    print(f"{h.chunk.id:8} p{h.chunk.page_start:<4} {h.chunk.chunk_type:15} score={h.score:.4f} dense#{h.dense_rank} bm25#{h.sparse_rank}  {h.chunk.section_label}")

intent: torque | content terms: ['brake', 'caliper', 'bolt']
d118-02  p636  torque_table    score=0.0610 dense#0 bm25#0  206-03 Front Disc Brake
d127-02  p652  torque_table    score=0.0600 dense#1 bm25#1  206-04 Rear Disc Brake
d107-02  p599  torque_table    score=0.0492 dense#4 bm25#2  206-00 Brake System — General Information
d136-02  p672  torque_table    score=0.0467 dense#10 bm25#3  206-05 Parking Brake and Actuation
d102-02  p578  torque_table    score=0.0412 dense#28 bm25#6  205-04 Front Drive Halfshafts
d147-01  p696  torque_table    score=0.0407 dense#24 bm25#10  206-06 Hydraulic Brake Actuation


## 5. Structured extraction -> JSON
`mode="auto"` uses the LLM when an API key is configured (and grounds every value against the source text); otherwise the deterministic table parser runs.

In [6]:
res = pipe.query(QUERY, mode="auto")
print("extractor used:", res.mode, res.llm_name, res.warnings)
print(res.to_json())

extractor used: rules  []
[
  {
    "component": "Brake caliper anchor plate bolts",
    "spec_type": "Torque",
    "value": "250",
    "unit": "Nm",
    "alt_values": [
      {
        "value": "184",
        "unit": "lb-ft"
      }
    ],
    "section": "206-03 Front Disc Brake",
    "page": 636,
    "source": "table",
    "verified": true,
    "evidence": "Brake caliper anchor plate bolts | 250 | 184 | —",
    "chunk_id": "d118-02",
    "also_found_in": []
  },
  {
    "component": "Brake caliper flow bolt",
    "spec_type": "Torque",
    "value": "35",
    "unit": "Nm",
    "alt_values": [
      {
        "value": "26",
        "unit": "lb-ft"
      }
    ],
    "section": "206-03 Front Disc Brake",
    "page": 636,
    "source": "table",
    "verified": true,
    "evidence": "Brake caliper flow bolt | 35 | 26 | —",
    "chunk_id": "d118-02",
    "also_found_in": [
      "206-04 Rear Disc Brake p.652"
    ]
  },
  {
    "component": "Brake caliper guide pin bolts",
    "spec_type":

## 6. The same results as CSV

In [7]:
print(res.to_csv())

component,spec_type,value,unit,alt_values,condition,notes,section,page,also_found_in,source,verified
Brake caliper anchor plate bolts,Torque,250,Nm,184 lb-ft,,,206-03 Front Disc Brake,636,,table,True
Brake caliper flow bolt,Torque,35,Nm,26 lb-ft,,,206-03 Front Disc Brake,636,206-04 Rear Disc Brake p.652,table,True
Brake caliper guide pin bolts,Torque,37,Nm,27 lb-ft,,,206-03 Front Disc Brake,636,,table,True
Brake caliper guide pin bolts,Torque,33,Nm,24 lb-ft,,,206-04 Rear Disc Brake,652,,table,True
Brake caliper support bracket bolts,Torque,150,Nm,111 lb-ft,,,206-04 Rear Disc Brake,652,,table,True



## 7. A few more queries
Fluid capacity, part numbers, fluid specs, dimensional limits and ranges - all from the same pipeline.

In [8]:
import pandas as pd
rows = []
for q in ["Brake fluid capacity", "Rear axle lubricant fill capacity", "Part number for shock absorber lower bolt",
          "Brake fluid specification", "Front brake disc minimum thickness", "Lower ball joint nut torque",
          "Pinion bearing rotational preload"]:
    for r in pipe.query(q, mode="auto").records:
        rows.append({"query": q, "component": r.component[:48], "type": r.spec_type, "value": r.value, "unit": r.unit,
                     "alt": "; ".join(f"{a.value} {a.unit}" for a in r.alt_values), "section": r.section, "page": r.page})
pd.DataFrame(rows)

,query,component,type,value,unit,alt,section,page
0,Brake fluid capacity,Motorcraft High Performance DOT 3 Motor Vehicle,Fluid Capacity,740,ml,1.56 pt,206-06 Hydraulic Brake Actuation,696
1,Brake fluid capacity,High Performance DOT 3 Motor Vehicle Brake Fluid,Fluid Capacity,1420,ml,3.0 pt,206-04 Rear Disc Brake,652
2,Rear axle lubricant fill capacity,Motorcraft SAE 80W-90 Premium Rear Axle Lubrican,Fluid Capacity,1.66,L,3.5 pt,205-03 Front Drive Axle/Differential,449
3,Rear axle lubricant fill capacity,Motorcraft SAE 75W-140 Synthetic Rear Axle Lubri,Fluid Capacity,2.84,L,6.0 pt,205-02A Rear Drive Axle/Differential — Ford 8....,258
4,Part number for shock absorber lower bolt,Shock absorber lower bolt,Part Number,W506545,NaN,,204-02 Rear Suspension,133
5,Part number for shock absorber lower bolt,Lower shock absorber bolts (2 required),Part Number,W506545,NaN,,205-02A Rear Drive Axle/Differential — Ford 8....,330
6,Part number for shock absorber lower bolt,Shock absorber lower nut,Part Number,W520214,NaN,,204-02 Rear Suspension,133
7,Part number for shock absorber lower bolt,Shock absorber shield bolt (3 required),Part Number,18A149-B,NaN,,204-02 Rear Suspension,133
8,Part number for shock absorber lower bolt,Lower shock absorber nuts (2 required),Part Number,W520214,NaN,,205-02A Rear Drive Axle/Differential — Ford 8....,330
9,Brake fluid specification,Motorcraft High Performance DOT 3 Motor Vehicle,Part Number,PM-1-C (US); CPM-1-C (Canada),NaN,,206-06 Hydraulic Brake Actuation,696


## 8. Evaluation
`eval/gold.json` holds expected values read straight from the PDF text. Two metrics: **retrieval hit-rate** (is the gold page among the
retrieved chunks?) and **extraction recall** (is the expected value returned?). One negative query checks that nothing is invented.

In [9]:
from specx.evaluate import evaluate
rep = evaluate(pipe, "../eval/gold.json", mode="auto")
print(rep.summary())
print("\n".join(rep.failures) or "no misses")

retrieval hit-rate : 26/26 = 100%
extraction recall  : 26/26 = 100%
negative queries   : 1/1 correctly empty
avg records/query  : 2.8
no misses
